# Notícias da Agência Brasil

Neste notebook montamos nossa própria base de dados: coletamos notícias do site da [Agência Brasil](https://agenciabrasil.ebc.com.br/), a agência pública de notícias da EBC, e usamos as técnicas de [02-primeiros-passos.ipynb](02-primeiros-passos.ipynb) para responder a uma pergunta:

**Sobre o que cada editoria da Agência Brasil está escrevendo agora?**

## Antes de raspar: o site permite?

Antes de coletar dados de um site, confira duas coisas:

1. **Os termos de uso.** A Agência Brasil diz que "as notícias podem ser reproduzidas desde que citada a fonte". Isso vale para o texto: algumas fotos são de outras agências e têm a reprodução proibida. Aqui usamos só o texto.
2. **O arquivo `robots.txt`**, que diz o que robôs podem acessar. O da Agência Brasil, em <https://agenciabrasil.ebc.com.br/robots.txt>, libera as notícias e pede um intervalo de 10 segundos entre um acesso e outro (`Crawl-delay: 10`).

Também é boa prática se identificar: o cabeçalho `User-Agent` diz ao site quem está acessando.

In [1]:
import string
import time
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import nltk
import pandas as pd
import requests
from bs4 import BeautifulSoup

nltk.download('stopwords', quiet=True)

HEADERS = {'User-Agent': 'jornalismo-de-dados (material didático; https://github.com/anaschwendler/jornalismo-de-dados)'}
DELAY = 10  # segundos entre um acesso e outro, como pede o robots.txt

## Passo 1: a lista de notícias (RSS)

Muitos sites de notícias publicam um *feed* RSS: um arquivo XML com as últimas notícias, feito para ser lido por programas. A Agência Brasil tem um feed para cada editoria, com as 10 notícias mais recentes. Dele, pegamos o link de cada notícia.

In [2]:
def feed_links(section):
    url = f'https://agenciabrasil.ebc.com.br/rss/{section}/feed.xml'
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    root = ET.fromstring(response.content)
    return [item.findtext('link') for item in root.findall('./channel/item')]

links = feed_links('politica')
links

['https://agenciabrasil.ebc.com.br/politica/noticia/2026-11/lula-garante-que-programas-sociais-serao-mantidos',
 'https://agenciabrasil.ebc.com.br/politica/noticia/2026-11/violencia-politica-disque-100-recebeu-189-denuncias-durante-eleicao',
 'https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/mte-recebe-43-mil-denuncias-de-assedio-eleitoral-em-tres-dias',
 'https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/aluno-e-ameacado-de-morte-no-df-por-opcao-politica',
 'https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/estudantes-relatam-violencia-relacionada-eleicao-presidencial',
 'https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/acao-popular-pede-suspensao-de-salario-do-senador-flavio-bolsonaro',
 'https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/vieira-encontra-rubio-nos-eua-apos-vitoria-de-lula-e-fuga-de-flavio',
 'https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/alcolumbre-marca-votacao-da-pec-da-seguranca-publica-para-terca-feira',
 'http

## Passo 2: o texto de uma notícia (Beautiful Soup)

A página de uma notícia é um arquivo HTML, com o texto misturado a menus, botões e propagandas. O Beautiful Soup lê o HTML e nos deixa pegar só as partes que interessam, usando *seletores CSS*.

Para descobrir o seletor de cada informação, abra uma notícia no navegador, clique com o botão direito no título e escolha **Inspecionar**: o navegador mostra a tag e a classe de cada elemento. Na Agência Brasil:

| Informação | Seletor |
|---|---|
| Título | `h1.titulo-materia` |
| Linha fina | `.linha-fina-noticia` |
| Autoria | `.autor-noticia` |
| Data | `meta[property="article:published_time"]` |
| Texto | `.conteudo-noticia > p`, os parágrafos dentro do corpo da notícia |

Se o site mudar o layout, esses seletores param de funcionar. Esse é o ponto fraco da raspagem de dados.

In [3]:
def text_of(soup, selector):
    element = soup.select_one(selector)
    return element.get_text(' ', strip=True) if element else ''


def get_article(url):
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    date = soup.select_one('meta[property="article:published_time"]')
    paragraphs = soup.select('.conteudo-noticia > p')
    return {
        'titulo': text_of(soup, 'h1.titulo-materia'),
        'linha_fina': text_of(soup, '.linha-fina-noticia'),
        'autoria': text_of(soup, '.autor-noticia'),
        'data': date['content'] if date else '',
        'url': url,
        'texto': '\n'.join(p.get_text(' ', strip=True) for p in paragraphs),
    }

time.sleep(DELAY)
article = get_article(links[0])
print(article['titulo'])
print(article['autoria'], '|', article['data'])
print()
print(article['texto'][:500])

Lula garante que programas sociais serão mantidos
Alex Rodrigues - Repórter da Agência Brasil | 2026-11-01T21:48:00-03:00

O presidente da República Luiz Inácio Lula da Silva garantiu, neste domingo (1º), que não vai extinguir nenhum programa social em seu próximo governo (2027-2030).
“O que pode acontecer é a gente criar novos programas sociais ou melhorar aqueles que já estão [funcionando]”, disse Lula em um vídeo publicado em seu perfil no Instagram .
Reeleito para seu quarto mandato à frente do Poder Executivo federal, Lula assegurou que tudo o que prometeu durante a campanha eleitoral será cumprido, sem a extin


## Passo 3: coletar várias editorias

Agora repetimos os passos 1 e 2 para cinco editorias. São até 50 notícias e, com 10 segundos entre um acesso e outro, a coleta leva uns 9 minutos.

Para não coletar tudo de novo cada vez que o notebook roda, salvamos o resultado em `data/agencia-brasil.csv`. Se o arquivo já existe, o notebook só lê o arquivo. Para pegar notícias mais recentes, apague o arquivo e rode de novo.

In [4]:
SECTIONS = ['politica', 'economia', 'saude', 'educacao', 'meio-ambiente']
CSV = Path('../data/agencia-brasil.csv')


def collect(sections):
    rows = []
    for section in sections:
        time.sleep(DELAY)
        for url in feed_links(section):
            time.sleep(DELAY)
            rows.append({'editoria': section, **get_article(url)})
    return pd.DataFrame(rows)

if CSV.exists():
    news = pd.read_csv(CSV)
else:
    news = collect(SECTIONS)
    news.to_csv(CSV, index=False)

news.groupby('editoria').size()

editoria
economia         10
educacao         10
meio-ambiente    10
politica         10
saude            10
dtype: int64

In [5]:
news[['editoria', 'data', 'titulo']].head(10)

,editoria,data,titulo
0,politica,2026-11-01T21:48:00-03:00,Lula garante que programas sociais serão mantidos
1,politica,2026-11-01T20:06:00-03:00,Denúncias de violência política quadriplicam e...
2,politica,2026-10-31T21:24:00-03:00,"MTE recebe 4,3 mil denúncias de assédio eleito..."
3,politica,2026-10-31T19:15:00-03:00,Aluno é ameaçado de morte no DF por opção polí...
4,politica,2026-10-31T18:15:00-03:00,Estudantes relatam violência relacionada à ele...
5,politica,2026-10-31T17:58:00-03:00,Ação Popular pede suspensão de salário do sena...
6,politica,2026-10-31T14:48:00-03:00,Vieira encontra Rubio nos EUA após vitória de ...
7,politica,2026-10-31T13:22:00-03:00,Alcolumbre marca votação da PEC da Segurança P...
8,politica,2026-10-30T13:46:00-03:00,Entidades repudiam listas de boicote por diver...
9,politica,2026-10-30T11:20:00-03:00,Ato de estudantes da UFF deu pontapé em mobili...


## Passo 4: as palavras mais frequentes por editoria

Usamos o que vimos no notebook 02: separamos as palavras com a expressão regular, passamos tudo para minúsculas e tiramos as stopwords. Também tiramos os números, que sozinhos não dizem do que a notícia trata.

In [6]:
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer

tokenizer = RegexpTokenizer(r"\w+(?:[-']\w+)*")
portuguese_stops = set(stopwords.words('portuguese'))


def words(text, stops):
    tokens = [token.lower() for token in tokenizer.tokenize(text)]
    return [token for token in tokens if token not in stops and not token.isdigit()]


def top_words(texts, stops, n=10):
    counter = Counter()
    for text in texts:
        counter.update(words(text, stops))
    return [word for word, _ in counter.most_common(n)]


def top_words_by_section(stops):
    return pd.DataFrame({
        section: top_words(news.loc[news['editoria'] == section, 'texto'], stops)
        for section in SECTIONS
    })

top_words_by_section(portuguese_stops)

,politica,economia,saude,educacao,meio-ambiente
0,estudantes,r,saúde,educação,região
1,lula,ano,diabetes,ensino,c
2,pública,segundo,doença,estudantes,sobre
3,brasil,mercado,retinopatia,enem,países
4,federal,brasil,anos,inep,sul
5,segurança,lote,agência,mec,brasil
6,presidente,produtos,diagnóstico,exame,biodiversidade
7,eleitoral,bilhões,diabética,cada,áreas
8,segundo,energia,ainda,sobre,tv
9,ministério,empresas,exame,nacional,perigo


Repare em palavras como "segundo", "sobre", "ainda" e "ano": aparecem em quase toda notícia e não dizem nada sobre o assunto. São as stopwords do jornalismo, que a lista do NLTK não conhece. Também aparecem letras soltas, como "r" (de R$) e "c" (de °C).

Não existe uma lista certa: olhe o resultado, acrescente palavras e rode de novo. Aqui juntamos a lista do NLTK, as letras soltas do alfabeto e algumas palavras comuns em notícias.

In [7]:
news_stops = portuguese_stops | set(string.ascii_lowercase) | {
    'segundo', 'sobre', 'ainda', 'cada', 'ano', 'anos', 'desde', 'brasil', 'agência',
    'disse', 'afirmou', 'pode', 'ser', 'vai', 'após', 'além',
}

top_words_by_section(news_stops)

,politica,economia,saude,educacao,meio-ambiente
0,estudantes,mercado,saúde,educação,região
1,lula,lote,diabetes,ensino,países
2,pública,produtos,doença,estudantes,sul
3,federal,bilhões,retinopatia,enem,biodiversidade
4,segurança,energia,diagnóstico,inep,áreas
5,presidente,empresas,diabética,mec,tv
6,eleitoral,milhões,exame,exame,perigo
7,ministério,transmissão,câncer,nacional,global
8,bolsonaro,nacional,cbo,médio,chuvas
9,flávio,controle,risco,dia,nacional


## O que encontramos

Na coleta salva neste notebook, feita em 2 de novembro de 2026, logo depois da eleição presidencial, a editoria de política é dominada pela eleição: "lula", "eleitoral", "bolsonaro" e "flávio" estão entre as palavras mais frequentes. Se você rodar de novo, verá outras notícias e outras palavras.

O método tem limites, e vale conhecê-los antes de transformar o resultado em notícia:

* **10 notícias por editoria é pouco.** Uma notícia longa pode dominar a lista: na coleta salva, "lote" aparece 18 vezes, todas em uma única notícia de economia, e "tv" aparece 13 vezes em uma única notícia de meio ambiente, sobre a programação da TV Brasil. Antes de tirar conclusões, conte também em quantas notícias cada palavra aparece.
* **Frequência não é o mesmo que importância.** Palavras como "nacional" aparecem em várias editorias. Para achar as palavras que distinguem uma editoria das outras, existe uma técnica chamada TF-IDF.
* **O RSS só traz as notícias mais recentes.** Para analisar um período maior, é preciso coletar várias vezes ao longo do tempo e juntar os resultados.